# Closing H1's Last Two Gaps: Standard-CNN Baseline + Noise-Robustness Test

Execution notebook built directly from `h1_completion_plan.docx` Section 8's
run order (Streams A and B). **Nothing in this notebook has been run yet** --
every cell below is ready to execute, but no number in it is real until you
actually run it against this project's real windowed N-CMAPSS data on Drive.

Source of truth for every function here:
- Cells marked **[verbatim from h1_completion_plan.docx]** are copied directly
  from that document's own code blocks.
- Cells marked **[new for this notebook]** fill a gap the plan named but did
  not give literal source for (mainly `benchmark_variant`) -- written to match
  the documented methodology, but not a quote from any document.
- `train_one_epoch`, `evaluate`, `build_dataloaders`, and every hyperparameter
  constant are now checked directly against the real `train.ipynb` (uploaded
  after this notebook's first draft) -- that draft had `loss_fn` missing from
  several calls (both functions require it) and a wrong `build_dataloaders`
  call shape; both are fixed below, and `pruning.py`/`quantization.py` had the
  same `loss_fn` bug fixed too (push the corrected versions before running
  the cells below that import them).
- Windowed data is now copied from Drive to local disk once, right after
  Section 0's mount, and read from there for the rest of the run -- found
  only by actually running this: a long run (the Section 6 loop plus QAT
  fallbacks, which can take a while on CPU) hit
  `OSError: [Errno 107] Transport endpoint is not connected` partway
  through, because Drive's FUSE mount silently dropped mid-run. Local disk
  can't suffer that disconnect.
- Sections 2 and 6 are now resumable across a reconnect: each checks Drive
  for a checkpoint from a previous run (the baseline model + its recorded
  val_f1_macro for Section 2; the recovered-FP32 and PTQ/QAT checkpoints
  per pruning ratio for Section 6) before doing any training, and loads it
  instead of redoing the work. "Run all" after a disconnect now only
  retrains whatever genuinely hasn't finished yet -- benchmarks
  (macro_f1/latency/size) are still always recomputed fresh, since those
  are cheap, inference-only, and should reflect the hardware you're
  actually running on. Every reload uses `map_location="cpu"` so a
  checkpoint saved on one runtime type (CPU or GPU) can always be loaded
  back on a different one later.
- Quantization (PTQ/QAT, Section 4.1 and the Section 6/8 benchmarking of
  their output) now always runs on CPU, regardless of what `device` the
  rest of the notebook trains/benchmarks FP32 models on -- found only by
  actually running this on a GPU runtime: PyTorch's eager-mode
  quantization (the "fbgemm" backend used here) has no CUDA kernels for
  its actual quantized ops, so a CUDA-resident model crashed as soon as
  quantization touched it. GPU still speeds up everything FP32
  (training, pruning recovery, FP32 benchmarking); only the
  already-quantized steps are pinned to CPU, since that's the only
  device they can actually run on either way.

Same verification discipline as every other module in this project: every
self-check below runs on synthetic data and must pass BEFORE the real-data
cell beneath it runs.

**Before running:** this notebook gets `edge_fault_dx` (the code --
`models/`, `training/`, `compression/`) by cloning the GitHub repo fresh
into the Colab runtime every time Section 0 runs below -- NOT from a
hand-uploaded copy on Drive. This replaces an earlier version of this
notebook that asked you to manually recreate the `edge_fault_dx/` folder
structure on Drive; that approach kept losing cells/files between manual
edits, so the clone is now the one source of truth for the code. Drive is
still used, but only for `DATA_DIR` -- your real windowed data and saved
checkpoints, which were never meant to be duplicated into a `src/`-style
tree.


## 0. Setup -- mount Drive (data only), clone the repo fresh (code), import this project's existing, already-proven modules unchanged

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# DATA_DIR: your real windowed data + saved checkpoints, already on Drive --
# this path is unchanged from how you're already using Drive.
REPO_ROOT_DRIVE = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace"
DATA_DIR = f"{REPO_ROOT_DRIVE}/data"

# Code comes from a fresh clone of the GitHub repo into the Colab runtime
# (NOT from Drive) -- this always matches whatever's actually pushed to
# GitHub, with no manual folder-structure upload step to get wrong.
REPO_URL = "https://github.com/iodawa/edge-ai-fault-diagnosis-aerospace.git"
REPO_DIR = "/content/edge-ai-fault-diagnosis-aerospace"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")

SRC_ROOT = f"{REPO_DIR}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)


Mounted at /content/drive


In [2]:
# REAL BUG, found by actually running this: with the windowed data read
# live off the Drive FUSE mount (f"{DATA_DIR}/windowed"), a long run (the
# full Section 6 compression loop plus QAT fallbacks takes a while on CPU)
# hit `OSError: [Errno 107] Transport endpoint is not connected` partway
# through -- Drive's FUSE mount silently dropped mid-run. This is a Colab
# infrastructure quirk, not a bug in build_dataloaders/WindowedFaultDataset:
# the long-lived Drive connection itself is what's unreliable, especially
# across a disconnect/reconnect like the one that happened here.
#
# Fix: copy the windowed data to local disk ONCE, right after mounting,
# and read from local disk for the rest of the run. Local reads can't
# suffer a Drive disconnect mid-training. This only copies the (small)
# windowed .npz files, not your checkpoints -- those are still saved
# straight to Drive at the end of each step, which is a single write each,
# not a long-lived read held open for hours.
import shutil

LOCAL_WINDOWED_DIR = "/content/windowed_local"
if not os.path.exists(LOCAL_WINDOWED_DIR):
    print("Copying windowed data from Drive to local disk once "
          "(avoids Drive FUSE disconnects during the long run that follows)...")
    shutil.copytree(f"{DATA_DIR}/windowed", LOCAL_WINDOWED_DIR)
    print("done:", LOCAL_WINDOWED_DIR)
else:
    print("local windowed data already present:", LOCAL_WINDOWED_DIR)


Copying windowed data from Drive to local disk once (avoids Drive FUSE disconnects during the long run that follows)...
done: /content/windowed_local


In [3]:
import time
from collections import OrderedDict

import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from scipy import stats

# Reused unchanged -- these must already exist at the paths below.
from edge_fault_dx.models.ma1dcnn import MA1DCNN, build_ma1dcnn, Conv1dSamePadding
from edge_fault_dx.training.train import (
    TRIAL_13_MODEL_CONFIG,
    TRIAL_13_TRAIN_CONFIG,
    DEFAULT_WINDOWED_DIR,
    N_EPOCHS,
    EARLY_STOPPING_PATIENCE,
    RANDOM_SEED,
    SCHEDULER_T_0,
    SCHEDULER_T_MULT,
    build_dataloaders,
    train_one_epoch,
    evaluate,
)

# evaluate()/train_one_epoch()'s real signatures (confirmed against train.ipynb) both
# take loss_fn as a required positional argument: train_one_epoch(model, loader,
# optimizer, loss_fn, device); evaluate(model, loader, loss_fn, device, threshold).
# run_training() itself uses nn.BCEWithLogitsLoss() -- reused unchanged here throughout.
loss_fn = nn.BCEWithLogitsLoss()
from edge_fault_dx.compression.pruning import (
    _l1_channel_importance,
    _topk_keep_indices,
    compute_pruned_channel_counts,
    recover_pruned_model,
)
from edge_fault_dx.compression.quantization import (
    quantize_model_ptq,
    quantize_model_qat,
    count_params_any,
    repair_missing_module_dicts,
    QAT_N_EPOCHS,
    QAT_PATIENCE,
    QAT_LR_SCALE,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# retention bar for StandardCNN, same proportional rule as MA1DCNN's
# (baseline macro-F1 - 5 points) -- filled in once Section 2 below
# actually trains standard_cnn_trial13 and prints its real baseline.
STANDARD_CNN_RETENTION_BAR = None


device: cuda


## 1. `models/standard_cnn.py` -- the architecture-only comparator (h1_completion_plan.docx Section 3)

**[verbatim from h1_completion_plan.docx]**, cleaned of line-wrap artifacts.
Identical to MA1DCNN except `ChannelAttentionModule` and
`ElementwiseAttentionModule` are removed -- conv3's output goes straight
into global average pooling. This is deliberate: "architecture only" is
H1's independent variable, so every other choice (channel sizes,
`kernel_size`, `fc_units`, `dropout`, training config) is copied from
`TRIAL_13_MODEL_CONFIG` / `TRIAL_13_TRAIN_CONFIG` unchanged.

Once this cell is proven against real data (Sections 2 and beyond), this
class belongs at `src/edge_fault_dx/models/standard_cnn.py` -- same
notebook-first-then-extract pattern `pruning.py`/`quantization.py` already
went through.


In [4]:
class StandardCNN(nn.Module):
    """3 conv blocks -> global average pooling -> FC head -> 5 logits.

    Identical to MA1DCNN (models/ma1dcnn.py) except it has no
    ChannelAttentionModule and no ElementwiseAttentionModule -- conv3's
    output goes straight into global_avg_pool. This is the whole point:
    H1 isolates "architecture only" as its independent variable, so every
    other choice (channel sizes, kernel_size, fc_units, dropout, training
    config) is copied from TRIAL_13_MODEL_CONFIG / TRIAL_13_TRAIN_CONFIG
    unchanged.
    """

    def __init__(
        self,
        n_input_channels,
        channel_1,
        channel_2,
        channel_3,
        kernel_size,
        fc_units,
        dropout,
        n_outputs,
        conv_dropout=0.0,
        use_batchnorm=False,
    ):
        super().__init__()
        self.conv1 = Conv1dSamePadding(n_input_channels, channel_1, kernel_size, conv_dropout, use_batchnorm)
        self.conv2 = Conv1dSamePadding(channel_1, channel_2, kernel_size, conv_dropout, use_batchnorm)
        self.conv3 = Conv1dSamePadding(channel_2, channel_3, kernel_size, conv_dropout, use_batchnorm)
        # no self.cam, no self.eam -- this is the entire architectural difference
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(channel_3, fc_units)
        self.fc_activation = nn.ReLU()
        self.output_layer = nn.Linear(fc_units, n_outputs)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        # (MA1DCNN inserts self.cam(x) then self.eam(x) here -- the only diff)
        x = self.global_avg_pool(x)
        x = x.view(x.shape[0], -1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.fc_activation(x)
        return self.output_layer(x)

    def predict_proba(self, x):
        with torch.no_grad():
            return torch.sigmoid(self.forward(x))


def build_standard_cnn(config):
    """Same config dict shape as build_ma1dcnn() -- TRIAL_13_MODEL_CONFIG
    works unchanged as input to both, which is exactly what keeps the
    comparison fair."""
    return StandardCNN(
        n_input_channels=config["n_input_channels"],
        channel_1=config["channel_1"],
        channel_2=config["channel_2"],
        channel_3=config["channel_3"],
        kernel_size=config["kernel_size"],
        fc_units=config["fc_units"],
        dropout=config["dropout"],
        n_outputs=config["n_outputs"],
        conv_dropout=config.get("conv_dropout", 0.0),
        use_batchnorm=config.get("use_batchnorm", False),
    )


### 1.1 Self-check -- parameter count matches the hand-derived prediction

h1_completion_plan.docx Section 3's hand count: conv1+conv2+conv3 identical
to MA1DCNN (55,904), cam+eam removed (-8,481), fc+output_layer identical
(2,149) -> **58,053 total**, 12.7% smaller than MA1DCNN's confirmed 66,534.
This must match exactly before trusting the class on real data.


In [5]:
standard_cnn = build_standard_cnn(TRIAL_13_MODEL_CONFIG)
standard_cnn_params = sum(p.numel() for p in standard_cnn.parameters())
print("StandardCNN total params:", standard_cnn_params)
assert standard_cnn_params == 58_053, (
    f"expected 58,053 per h1_completion_plan.docx Section 3, got {standard_cnn_params} -- "
    "check TRIAL_13_MODEL_CONFIG hasn't drifted from the values that document used "
    "(channel_1=32, channel_2=64, channel_3=128, kernel_size=5, fc_units=16, "
    "n_input_channels=28, n_outputs=5)."
)

# shape / probability-range self-check, same pattern ma1dcnn.ipynb used
dummy = torch.randn(4, TRIAL_13_MODEL_CONFIG["n_input_channels"], 50)
probs = standard_cnn.predict_proba(dummy)
assert probs.shape == (4, TRIAL_13_MODEL_CONFIG["n_outputs"])
assert torch.all((probs >= 0) & (probs <= 1))
print("Self-check PASSED: params == 58,053, output shape and probability range correct.")


StandardCNN total params: 58053
Self-check PASSED: params == 58,053, output shape and probability range correct.


## 2. Train the baseline (h1_completion_plan.docx Section 4)

The plan's own recommended path is a one-line `train.py` edit -- a
`build_fn=build_ma1dcnn` parameter on `run_training()` (default preserves
every existing call site). Now that `train.ipynb`'s real source is
available, that edit really is as small as the plan says: `run_training()`
(Section 9 there) hard-codes `model = build_ma1dcnn(model_config).to(device)`
as its one architecture-specific line -- adding a `build_fn=build_ma1dcnn`
parameter and using it there is the whole change.

**[new for this notebook]** `run_training_like()` below avoids editing
`train.py` directly anyway, so this notebook doesn't touch a file it
can't run end to end itself: it reuses `train_one_epoch`, `evaluate`, and
`build_dataloaders` exactly as imported, and now mirrors `run_training()`'s
real loop field-for-field (`torch.manual_seed(RANDOM_SEED)`, the same
`CosineAnnealingWarmRestarts(T_0=10, T_mult=2)`, the same
`N_EPOCHS=30` / `EARLY_STOPPING_PATIENCE=5`, the same strictly-greater
best-checkpoint rule) -- confirmed against train.ipynb Section 9, not just
assumed. If you'd rather apply the plan's literal `build_fn` edit to
`train.py` instead, do that and call
`run_training(..., build_fn=build_standard_cnn)` in place of this cell --
both now produce the same result by construction.


In [6]:
def run_training_like(build_fn, model_config, train_config, train_loader, val_loader,
                       device, n_epochs=N_EPOCHS, patience=EARLY_STOPPING_PATIENCE):
    """[new for this notebook] Same effect as run_training(build_fn=...)
    would have, built only from the pieces already confirmed
    architecture-agnostic: build_dataloaders, train_one_epoch, evaluate.
    Mirrors run_training()'s real loop (train.ipynb Section 9)
    field-for-field: same torch.manual_seed(RANDOM_SEED), same
    CosineAnnealingWarmRestarts(T_0, T_mult), same N_EPOCHS/patience
    defaults, same strictly-greater best-checkpoint rule.
    """
    torch.manual_seed(RANDOM_SEED)
    model = build_fn(model_config).to(device)
    optimizer = Adam(model.parameters(), lr=train_config["learning_rate"],
                      weight_decay=train_config["weight_decay"])
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=SCHEDULER_T_0, T_mult=SCHEDULER_T_MULT)

    best_f1 = -1.0
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, n_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_metrics = evaluate(model, val_loader, loss_fn, device, threshold=train_config["threshold"])
        scheduler.step()
        history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})
        print(f"epoch {epoch:2d}  train_loss={train_loss:.4f}  val_f1_macro={val_metrics['f1_macro']:.4f}")

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"early stopping at epoch {epoch} (patience={patience})")
                break

    model.load_state_dict(best_state)
    return model, history


In [7]:
# build_dataloaders()'s real signature (confirmed against train.ipynb):
# build_dataloaders(windowed_dir=DEFAULT_WINDOWED_DIR, batch_size=..., files=None, num_workers=0)
# -- it does NOT take a train_config dict directly.
#
# NOTE: DEFAULT_WINDOWED_DIR as committed in GitHub's train.py is a relative
# path ("data/windowed") -- it isn't written to know about any one person's
# Drive location. Point this at the local copy made in Section 0
# (LOCAL_WINDOWED_DIR), not the live Drive path -- see that cell's note on
# the real Drive-FUSE-disconnect bug this avoids.
train_loader, val_loader = build_dataloaders(
    windowed_dir=LOCAL_WINDOWED_DIR,
    batch_size=TRIAL_13_TRAIN_CONFIG["batch_size"],
    files=None,
)

# Resumable: this training run is the single most expensive step in the
# whole notebook (up to N_EPOCHS=30 on CPU), and a reconnect means
# re-running every cell from the top -- without this check, that means
# retraining from scratch every single time, even though the result is
# saved to Drive already. So: if a checkpoint from a previous run is
# already sitting on Drive, load it and skip training entirely. Only
# train for real when there is nothing to resume from.
import json

STANDARD_CNN_CKPT_PATH = f"{DATA_DIR}/standard_cnn_trial13.pt"
STANDARD_CNN_BASELINE_F1_PATH = f"{DATA_DIR}/standard_cnn_trial13_baseline_f1.json"

if os.path.exists(STANDARD_CNN_CKPT_PATH) and os.path.exists(STANDARD_CNN_BASELINE_F1_PATH):
    print("Found an existing StandardCNN baseline checkpoint on Drive -- "
          "loading it instead of retraining.")
    # map_location="cpu": a checkpoint saved on a CUDA runtime has CUDA
    # tensors inside it by default -- loading it straight on a later
    # CPU-only runtime would crash. Load to CPU and let benchmark_variant()
    # / prune_standard_cnn()'s callers move it to `device` as needed.
    standard_cnn_trained = torch.load(STANDARD_CNN_CKPT_PATH, weights_only=False, map_location="cpu")
    with open(STANDARD_CNN_BASELINE_F1_PATH) as f:
        standard_cnn_baseline_f1 = json.load(f)["best_val_f1_macro"]
    print("loaded best val_f1_macro:", standard_cnn_baseline_f1)
else:
    standard_cnn_trained, standard_cnn_history = run_training_like(
        build_standard_cnn,
        TRIAL_13_MODEL_CONFIG,
        TRIAL_13_TRAIN_CONFIG,
        train_loader,
        val_loader,
        device,
    )

    standard_cnn_baseline_f1 = max(h["f1_macro"] for h in standard_cnn_history)
    print("\nstandard_cnn_trial13 best val_f1_macro:", standard_cnn_baseline_f1)

    torch.save(standard_cnn_trained, STANDARD_CNN_CKPT_PATH)
    with open(STANDARD_CNN_BASELINE_F1_PATH, "w") as f:
        json.dump({"best_val_f1_macro": standard_cnn_baseline_f1}, f)
    print("saved:", STANDARD_CNN_CKPT_PATH, "and", STANDARD_CNN_BASELINE_F1_PATH)

STANDARD_CNN_RETENTION_BAR = standard_cnn_baseline_f1 - 0.05
print("StandardCNN retention bar (baseline - 5 pts):", STANDARD_CNN_RETENTION_BAR)


Found an existing StandardCNN baseline checkpoint on Drive -- loading it instead of retraining.
loaded best val_f1_macro: 0.831868016309856
StandardCNN retention bar (baseline - 5 pts): 0.7818680163098559


## 3. `prune_standard_cnn` (h1_completion_plan.docx Section 5.1)

**[verbatim structure from h1_completion_plan.docx]**, with the elided
`conv1 -> conv2 -> conv3` and `fc` weight-copy lines filled in from
`prune_ma1dcnn`'s own Section 5.1 logic (now in `pruning.py`) -- the plan
explicitly says this is "exactly prune_ma1dcnn's Section 5.1 ... logic,"
just without the CAM/EAM branch, since StandardCNN has neither module.


In [8]:
def prune_standard_cnn(model, model_config, prune_ratio):
    """Structured channel pruning of conv1/conv2/conv3 by L1-norm,
    rebuilding conv2's input, conv3's input, and fc's input to match --
    the same three dependent-layer rebuilds prune_ma1dcnn does for
    MA1DCNN's conv stack, with no CAM/EAM step because StandardCNN has
    neither module (h1_completion_plan.docx Section 5.1).
    """
    c1, c2, c3 = compute_pruned_channel_counts(
        model_config["channel_1"], model_config["channel_2"], model_config["channel_3"], prune_ratio,
    )

    pruned_config = dict(model_config)
    pruned_config.update(channel_1=c1, channel_2=c2, channel_3=c3)
    pruned_model = build_standard_cnn(pruned_config)

    keep1 = _topk_keep_indices(_l1_channel_importance(model.conv1.conv.weight), c1)
    keep2 = _topk_keep_indices(_l1_channel_importance(model.conv2.conv.weight), c2)
    keep3 = _topk_keep_indices(_l1_channel_importance(model.conv3.conv.weight), c3)

    with torch.no_grad():
        pruned_model.conv1.conv.weight.copy_(model.conv1.conv.weight[keep1, :, :])
        pruned_model.conv1.conv.bias.copy_(model.conv1.conv.bias[keep1])

        pruned_model.conv2.conv.weight.copy_(model.conv2.conv.weight[keep2, :, :][:, keep1, :])
        pruned_model.conv2.conv.bias.copy_(model.conv2.conv.bias[keep2])

        pruned_model.conv3.conv.weight.copy_(model.conv3.conv.weight[keep3, :, :][:, keep2, :])
        pruned_model.conv3.conv.bias.copy_(model.conv3.conv.bias[keep3])

        # fc: Linear(channel_3, fc_units) -- in_features (columns) -> keep3, same as prune_ma1dcnn's fc step
        pruned_model.fc.weight.copy_(model.fc.weight[:, keep3])
        pruned_model.fc.bias.copy_(model.fc.bias)

        # output_layer: never pruned -- neither dimension depends on a pruned channel count
        pruned_model.output_layer.weight.copy_(model.output_layer.weight)
        pruned_model.output_layer.bias.copy_(model.output_layer.bias)

    return pruned_model


### 3.1 Self-check -- lossless at prune_ratio = 0.0 (same discipline as pruning_code_explained.docx Section 7.1)

In [9]:
probe_model = build_standard_cnn(TRIAL_13_MODEL_CONFIG)
probe_model.eval()
probe_input = torch.randn(4, TRIAL_13_MODEL_CONFIG["n_input_channels"], 50)

with torch.no_grad():
    out_original = probe_model(probe_input)

pruned_at_zero = prune_standard_cnn(probe_model, TRIAL_13_MODEL_CONFIG, prune_ratio=0.0)
pruned_at_zero.eval()
with torch.no_grad():
    out_pruned = pruned_at_zero(probe_input)

assert torch.allclose(out_original, out_pruned, atol=1e-5), "lossless-at-ratio-0.0 check FAILED"
print("Self-check PASSED: prune_standard_cnn is lossless at prune_ratio=0.0.")


Self-check PASSED: prune_standard_cnn is lossless at prune_ratio=0.0.


## 4. `QuantizedStandardCNNTrunk` (h1_completion_plan.docx Section 5.2)

**[verbatim structure from h1_completion_plan.docx]**, elided
`global_avg_pool` / `dropout` / `fc_activation` assignments filled in to
mirror `QuantizedConvTrunk`'s own convention exactly. Deliberately scoped
to conv1/conv2/conv3 only -- quantizing StandardCNN's FC head too, when
MA1DCNN's stays FP32, would make "how much of the model is INT8" an
uncontrolled second difference between the two pipelines (Section 5.2's
own reasoning).


In [10]:
class QuantizedStandardCNNTrunk(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.conv1, self.conv2, self.conv3 = model.conv1, model.conv2, model.conv3
        self.dequant = torch.quantization.DeQuantStub()
        self.fc, self.output_layer = model.fc, model.output_layer   # unquantized, by reference
        self.global_avg_pool = model.global_avg_pool
        self.dropout = model.dropout
        self.fc_activation = model.fc_activation

    def forward(self, x):
        x = self.quant(x)
        x = self.conv3(self.conv2(self.conv1(x)))
        x = self.dequant(x)                       # INT8 -> FP32 before global_avg_pool
        return self.output_layer(self.fc_activation(self.fc(self.dropout(
            self.global_avg_pool(x).view(x.shape[0], -1)))))


### 4.1 PTQ / QAT for StandardCNN -- the real bug this notebook hit

**[new for this notebook]**, found only by actually running Section 6
below: `quantize_model_ptq`/`quantize_model_qat`, imported unchanged from
`quantization.py`, both hardcode `QuantizedConvTrunk` inside themselves --
which assumes `model.cam`/`model.eam` exist, since it was written only for
MA1DCNN. Calling either function on a `StandardCNN` instance fails
immediately with `AttributeError: 'StandardCNN' object has no attribute
'cam'`. `QuantizedStandardCNNTrunk` above exists for exactly this reason,
but nothing before this point ever called it.

The two functions below are StandardCNN's counterparts, following
`quantize_model_ptq`/`quantize_model_qat`'s exact procedure (same fbgemm
qconfig, same prepare/calibrate/convert for PTQ, same prepare_qat/
FakeQuantize/fine-tune/convert for QAT, same `QAT_N_EPOCHS`/`QAT_PATIENCE`/
`QAT_LR_SCALE` budget, reused unchanged from `quantization.py`) -- the only
real difference is wrapping in `QuantizedStandardCNNTrunk` instead of
`QuantizedConvTrunk`, and freezing `fc`/`output_layer` only during QAT
(StandardCNN has no `cam`/`eam` to also freeze).

**A second, real bug, also found only by running Section 6**:
`NotImplementedError: Could not run 'quantized::linear' with arguments
from the 'CPU' backend`. `QuantizedStandardCNNTrunk.__init__` reuses
`fc`/`output_layer` "unquantized, by reference" per its own docstring,
but never actually enforces that -- `torch.quantization.prepare()`
recursively propagates the trunk's fbgemm qconfig to every child module
that doesn't set its own, `fc`/`output_layer` included. `convert()` then
silently turns them into real quantized `nn.Linear` layers, and
`forward()` immediately breaks them by feeding an FP32 tensor right after
`dequant()`. Fixed below by explicitly setting `qconfig = None` on
everything after `dequant()` before calling `prepare()`/`prepare_qat()`.

This same gap is structurally present in `QuantizedConvTrunk` in the
already-pushed `quantization.py` -- it reuses `cam`/`eam`/`fc`/
`output_layer` "unquantized, by reference" the same way, with the same
missing `qconfig = None`.

**A third real bug, confirmed by actually hitting it** (not just a
suspected parallel this time): `ValueError: optimizer got an empty
parameter list`, when PTQ misses the retention bar and the QAT fallback
runs next. `QuantizedStandardCNNTrunk.__init__` assigns
`self.conv1, self.conv2, self.conv3 = model.conv1, model.conv2, model.conv3`
-- a reference, not a copy. `torch.quantization.convert(wrapped,
inplace=True)` inside PTQ therefore mutates those same submodules on the
caller's own model, silently swapping its FP32 trunk for real INT8
modules. When the QAT fallback then wraps that same, now-already-quantized
model a second time, there are no ordinary trainable parameters left to
hand `Adam(...)`. Fixed below by deep-copying `model` before wrapping, so
PTQ and QAT each quantize their own independent FP32 copy.

`QuantizedConvTrunk` in `quantization.py` has the identical by-reference
assignment for `conv1`/`conv2`/`conv3` (and `cam`/`eam`/`fc`/
`output_layer`), and Section 6's loop structure here mirrors exactly how
that file's own docstring describes Phase 4's real run: PTQ first, QAT
only if PTQ misses the bar, on the same `recovered` object either way.
That means the documented real numbers for MA1DCNN (0.7768 PTQ then 0.8385
QAT at 20%, missing the bar both times at 40%/60% too) were produced by
code that -- as currently pushed -- would hit this exact same
"optimizer got an empty parameter list" crash on the QAT fallback step.
Worth fixing and re-verifying there too before leaning on those numbers
again; not changed here since that file wasn't touched as part of
debugging this notebook.

**A fourth real bug, found only by actually running this on a GPU
runtime**: `RuntimeError: Input type (torch.FloatTensor) and weight type
(torch.cuda.FloatTensor) should be the same`. Eager-mode PyTorch
quantization (the "fbgemm" backend used here) only has CPU kernels for
its actual quantized ops -- `model` arrives already moved to `device` by
`benchmark_variant()`, so on CUDA this wraps a CUDA model and then feeds
it plain-CPU calibration batches. Fixed below by always quantizing (and
benchmarking the quantized result) on CPU, independent of whatever
`device` the rest of the notebook trains/benchmarks FP32 models on. This
is also structurally present in `quantization.py`'s `quantize_model_ptq`/
`quantize_model_qat` for the same reason -- not changed there, same as
the other three.


In [11]:
import copy

def quantize_standard_cnn_ptq(model, calib_loader, n_calib_batches=200):
    """StandardCNN counterpart to quantize_model_ptq() (quantization.py) --
    identical procedure, wrapped in QuantizedStandardCNNTrunk instead of
    QuantizedConvTrunk since StandardCNN has no cam/eam to carry through
    unquantized.

    REAL BUG, found by actually running this: QuantizedStandardCNNTrunk
    reuses conv1/conv2/conv3 by REFERENCE (self.conv1 = model.conv1, not a
    copy). torch.quantization.convert(wrapped, inplace=True) therefore
    mutates those same submodules on the caller's own `model` object, not
    just on `wrapped` -- silently replacing `model`'s FP32 conv trunk with
    real INT8 modules. If the caller (Section 6 below) later reuses that
    same `model` for QAT after PTQ missed the retention bar, QAT wraps an
    already-quantized trunk with no ordinary trainable parameters left,
    and Adam(...) raises "optimizer got an empty parameter list". Fixed by
    deep-copying model before wrapping, so PTQ and QAT each get their own
    independent, still-FP32 copy of the recovered checkpoint to quantize.

    A FOURTH real bug, found only by actually running this on a GPU
    runtime: eager-mode PyTorch quantization (the "fbgemm" backend used
    here) only has CPU kernels for its actual quantized ops -- there is
    no CUDA implementation of quantized::conv1d / quantized::linear.
    `model` arrives here already moved to `device` (benchmark_variant()
    moves `recovered` to `device` before this is called), so on a CUDA
    runtime QuantizedStandardCNNTrunk wraps a CUDA model while the
    calibration batches from calib_loader are plain CPU tensors, which
    crashes with `RuntimeError: Input type (torch.FloatTensor) and weight
    type (torch.cuda.FloatTensor) should be the same`. Moving the batches
    to CUDA instead wouldn't actually fix this -- convert() would then
    try to build quantized ops on a CUDA module, which fbgemm doesn't
    support at all. Fixed by always quantizing on CPU here, regardless of
    what `device` the rest of the notebook trains/benchmarks FP32 models
    on -- the calibration batches are left alone since DataLoader already
    yields plain CPU tensors.
    """
    wrapped = QuantizedStandardCNNTrunk(copy.deepcopy(model).to("cpu"))
    wrapped.eval()
    wrapped.qconfig = torch.quantization.get_default_qconfig("fbgemm")
    # REAL BUG, found by actually running this: torch.quantization.prepare()
    # recursively propagates wrapped.qconfig to every child module that
    # doesn't set its own -- including fc/output_layer, even though the
    # class docstring says they're "unquantized, by reference". Without
    # this, convert() silently turns them into real quantized nn.Linear,
    # and forward() then feeds them an FP32 tensor right after dequant(),
    # which crashes with NotImplementedError: Could not run
    # 'quantized::linear' with arguments from the 'CPU' backend. Setting
    # qconfig=None on everything after dequant() opts them out for real.
    for name in ("global_avg_pool", "dropout", "fc", "fc_activation", "output_layer"):
        getattr(wrapped, name).qconfig = None
    torch.quantization.prepare(wrapped, inplace=True)

    with torch.no_grad():
        for i, (X, _) in enumerate(calib_loader):
            if i >= n_calib_batches:
                break
            wrapped(X.to("cpu"))

    torch.quantization.convert(wrapped, inplace=True)
    return wrapped


def quantize_standard_cnn_qat(model, train_config, train_loader, val_loader, device,
                               n_epochs=QAT_N_EPOCHS, patience=QAT_PATIENCE):
    """StandardCNN counterpart to quantize_model_qat() -- identical
    prepare_qat/FakeQuantize/fine-tune/convert procedure. Only fc and
    output_layer are frozen during the trunk fine-tune (one fewer module
    than MA1DCNN's version, since StandardCNN has no cam/eam).

    Same deepcopy fix as quantize_standard_cnn_ptq() -- without it, this
    function mutates the caller's `model` in place too, same as PTQ does.

    Same CPU-only fix as quantize_standard_cnn_ptq() too, and for the
    same reason: this still ends in torch.quantization.convert(), which
    needs fbgemm's CPU-only quantized ops, so the fine-tune leading up to
    it has to happen on CPU as well or convert() would be converting a
    CUDA module fbgemm can't handle. The `device` parameter is kept in
    the signature for call-site compatibility but intentionally NOT used
    to place anything here -- QAT always runs on CPU, independent of
    whatever device the rest of the notebook trains FP32 models on.
    """
    wrapped = QuantizedStandardCNNTrunk(copy.deepcopy(model).to("cpu"))
    wrapped.train()
    wrapped.qconfig = torch.quantization.get_default_qat_qconfig("fbgemm")
    # Same real bug/fix as quantize_standard_cnn_ptq() above -- exclude
    # everything after dequant() from the qconfig prepare_qat() would
    # otherwise propagate to it.
    for name in ("global_avg_pool", "dropout", "fc", "fc_activation", "output_layer"):
        getattr(wrapped, name).qconfig = None
    torch.quantization.prepare_qat(wrapped, inplace=True)

    for name in ("fc", "output_layer"):
        for p in getattr(wrapped, name).parameters():
            p.requires_grad_(False)

    optimizer = Adam(
        (p for p in wrapped.parameters() if p.requires_grad),
        lr=train_config["learning_rate"] * QAT_LR_SCALE,
    )

    best_f1 = -1.0
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, n_epochs + 1):
        # "cpu" here, not `device` -- see the CPU-only note above.
        train_one_epoch(wrapped, train_loader, optimizer, loss_fn, "cpu")
        val_metrics = evaluate(wrapped, val_loader, loss_fn, "cpu", threshold=train_config["threshold"])
        history.append({"epoch": epoch, **val_metrics})

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.clone() for k, v in wrapped.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    wrapped.load_state_dict(best_state)
    wrapped.eval()
    torch.quantization.convert(wrapped, inplace=True)
    return wrapped, history


## 5. Benchmarking -- params / size / CPU latency

**[new for this notebook]** `benchmark_variant`'s literal source isn't
quoted in any document available when this notebook was built -- only its
methodology (pruning_code_explained.docx Section 10): batch size 1, 10
warm-up runs excluded, >=100 timed runs (200 used throughout this
project), macro-F1 via `evaluate()` unchanged, params via `count_params_any`
(quantization-safe), size via a real serialized-file byte count.


In [12]:
def benchmark_variant(model, val_loader, device, threshold, n_warmup=10, n_timed=200):
    """[new for this notebook] params / size (MB) / CPU latency p50,p95,p99 (ms),
    following pruning_code_explained.docx Section 10's documented methodology.
    """
    model.eval().to(device)

    val_metrics = evaluate(model, val_loader, loss_fn, device, threshold=threshold)

    n_params = count_params_any(model)

    import tempfile, os
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        torch.save(model, f.name)
        size_mb = os.path.getsize(f.name) / (1024 * 1024)
    os.unlink(f.name)

    single_input = next(iter(val_loader))[0][:1].to(device)
    with torch.no_grad():
        for _ in range(n_warmup):
            model(single_input)

        timings_ms = []
        for _ in range(n_timed):
            if device == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(single_input)
            if device == "cuda":
                torch.cuda.synchronize()
            timings_ms.append((time.perf_counter() - t0) * 1000.0)

    timings_ms = np.array(timings_ms)
    p50, p95, p99 = np.percentile(timings_ms, [50, 95, 99])

    return {
        "macro_f1": val_metrics["f1_macro"],
        "params": n_params,
        "size_mb": size_mb,
        "latency_p50_ms": p50,
        "latency_p95_ms": p95,
        "latency_p99_ms": p99,
    }


## 6. Run StandardCNN through the full Phase 4 pipeline (h1_completion_plan.docx Section 8, step 4)

Prune at 20/40/60%, recover, benchmark (Steps 1-4 equivalent) -- then PTQ,
check against `STANDARD_CNN_RETENTION_BAR`, fall back to QAT only if PTQ
misses it (the same real-result-first rule that decided PTQ-vs-QAT for
MA1DCNN, not assumed in advance).


In [13]:
# Resumable, per ratio: recover_pruned_model's fine-tune and QAT's
# fine-tune are the two expensive, training-based steps in this loop
# (PTQ itself is just a few hundred calibration forward passes -- cheap).
# On a reconnect-and-run-all, re-running this cell from scratch means
# redoing all of that training again even though the results are already
# saved to Drive. So each ratio now checks Drive first:
#   - a saved recovered-FP32 checkpoint skips prune_standard_cnn +
#     recover_pruned_model (the first expensive step);
#   - a saved QAT checkpoint means that ratio is fully done -- skip
#     straight to re-benchmarking it, no PTQ/QAT work at all;
#   - a saved PTQ checkpoint (no QAT one) skips PTQ's calibration, and
#     the verdict against STANDARD_CNN_RETENTION_BAR is still re-checked
#     fresh each time (cheap -- just a benchmark_variant() call) in case
#     QAT still needs to run for it.
# Benchmarks themselves (macro_f1/params/size/latency) are always
# recomputed fresh rather than cached, since they're inference-only and
# cheap, and latency specifically should reflect the hardware you're
# actually running on right now.
standard_cnn_results = {}

for ratio in (0.20, 0.40, 0.60):
    print(f"\n=== StandardCNN, prune_ratio={ratio:.0%} ===")
    tag = int(ratio * 100)
    recovered_path = f"{DATA_DIR}/standard_cnn_trial13_pruned{tag}_recovered_fp32.pt"
    ptq_path = f"{DATA_DIR}/standard_cnn_trial13_pruned{tag}_ptq.pt"
    qat_path = f"{DATA_DIR}/standard_cnn_trial13_pruned{tag}_qat.pt"

    # map_location="cpu" on every reload below: a checkpoint saved during a
    # CUDA session has CUDA tensors inside it by default, and loading that
    # straight on a later CPU-only runtime (or vice versa) would either
    # crash outright (deserializing CUDA tensors with no GPU present) or
    # silently leave a model on the wrong device. Forcing CPU on load and
    # letting benchmark_variant()/recover_pruned_model() move FP32 models
    # to `device` themselves keeps this notebook safe to resume on a
    # different runtime type than the one that produced the checkpoint.
    if os.path.exists(recovered_path):
        print(f"Found an existing recovered FP32 checkpoint for {ratio:.0%} on Drive -- "
              "loading it instead of re-pruning/re-fine-tuning.")
        recovered = torch.load(recovered_path, weights_only=False, map_location="cpu")
    else:
        pruned = prune_standard_cnn(standard_cnn_trained, TRIAL_13_MODEL_CONFIG, ratio)
        recovered, recovery_history = recover_pruned_model(
            pruned, TRIAL_13_TRAIN_CONFIG, train_loader, val_loader, device,
        )
        torch.save(recovered, recovered_path)

    fp32_bench = benchmark_variant(recovered, val_loader, device, TRIAL_13_TRAIN_CONFIG["threshold"])
    print("FP32 recovered:", fp32_bench)

    chosen = {"fp32": fp32_bench, "ratio": ratio, "recovered_model": recovered}

    # "cpu" below, not `device`, for every quantized (post-convert) model --
    # see Section 4.1's note on the real CUDA-vs-fbgemm bug this avoids.
    if os.path.exists(qat_path):
        print(f"Found an existing QAT checkpoint for {ratio:.0%} on Drive -- "
              "this ratio is already done, skipping PTQ/QAT entirely.")
        qat_model = torch.load(qat_path, weights_only=False, map_location="cpu")
        qat_bench = benchmark_variant(qat_model, val_loader, "cpu", TRIAL_13_TRAIN_CONFIG["threshold"])
        print("QAT (loaded from Drive):", qat_bench)
        chosen.update(qat=qat_bench, method="qat")
        best_model = qat_model
    else:
        if os.path.exists(ptq_path):
            print(f"Found an existing PTQ checkpoint for {ratio:.0%} on Drive -- "
                  "skipping PTQ calibration.")
            ptq_model = torch.load(ptq_path, weights_only=False, map_location="cpu")
        else:
            # quantize_standard_cnn_ptq (Section 4.1), NOT quantize_model_ptq --
            # the latter hardcodes QuantizedConvTrunk, which needs
            # model.cam/model.eam and fails on StandardCNN (see Section
            # 4.1's note on the real bug this hit on first run).
            ptq_model = quantize_standard_cnn_ptq(recovered, train_loader)
            torch.save(ptq_model, ptq_path)

        # TODO: resweep_threshold needs threshold_sweep.py's real sweep() signature --
        # confirm this call matches it once threshold_sweep.py is imported successfully.
        ptq_threshold = resweep_threshold(ptq_model, val_loader, "cpu") if False else TRIAL_13_TRAIN_CONFIG["threshold"]
        ptq_bench = benchmark_variant(ptq_model, val_loader, "cpu", ptq_threshold)
        print("PTQ:", ptq_bench)

        verdict = "PASSED" if ptq_bench["macro_f1"] >= STANDARD_CNN_RETENTION_BAR else "BELOW RETENTION BAR"
        print(f"PTQ verdict ({ptq_bench['macro_f1']:.4f} vs bar {STANDARD_CNN_RETENTION_BAR:.4f}): {verdict}")
        chosen.update(ptq=ptq_bench, method="ptq")

        if verdict == "BELOW RETENTION BAR":
            print("PTQ missed the bar -- building QAT fallback (same trigger rule as MA1DCNN's Step 5).")
            qat_model, qat_history = quantize_standard_cnn_qat(
                recovered, TRIAL_13_TRAIN_CONFIG, train_loader, val_loader, device,
            )
            qat_threshold = TRIAL_13_TRAIN_CONFIG["threshold"]  # TODO: resweep once threshold_sweep.py is wired in
            qat_bench = benchmark_variant(qat_model, val_loader, "cpu", qat_threshold)
            print("QAT:", qat_bench)
            qat_verdict = "PASSED" if qat_bench["macro_f1"] >= STANDARD_CNN_RETENTION_BAR else "BELOW RETENTION BAR"
            print(f"QAT verdict ({qat_bench['macro_f1']:.4f} vs bar {STANDARD_CNN_RETENTION_BAR:.4f}): {qat_verdict}")
            chosen.update(qat=qat_bench, method="qat" if qat_verdict == "PASSED" else "ptq")
            if qat_verdict == "PASSED":
                torch.save(qat_model, qat_path)
                best_model = qat_model
            else:
                best_model = ptq_model
        else:
            best_model = ptq_model

    chosen["best_model"] = best_model
    standard_cnn_results[ratio] = chosen

    torch.save(best_model, f"{DATA_DIR}/standard_cnn_trial13_pruned{tag}_quantized.pt")

print("\nStandardCNN compression pipeline complete.")
for ratio, r in standard_cnn_results.items():
    print(f"  {ratio:.0%}: method={r['method']}  macro_f1={r.get(r['method'], {}).get('macro_f1')}")



=== StandardCNN, prune_ratio=20% ===
Found an existing recovered FP32 checkpoint for 20% on Drive -- loading it instead of re-pruning/re-fine-tuning.
FP32 recovered: {'macro_f1': 0.834464487740231, 'params': 38192, 'size_mb': 0.15448474884033203, 'latency_p50_ms': np.float64(0.12339499994595826), 'latency_p95_ms': np.float64(0.13221750006096045), 'latency_p99_ms': np.float64(0.13485379996382105)}


/tmp/ipykernel_1780/4132578582.py:52: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare(wrapped, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of 

PTQ: {'macro_f1': 0.790884303847841, 'params': 38192, 'size_mb': 0.05506610870361328, 'latency_p50_ms': np.float64(0.20919999997204286), 'latency_p95_ms': np.float64(0.23689200000376318), 'latency_p99_ms': np.float64(0.3682946000242281)}
PTQ verdict (0.7909 vs bar 0.7819): PASSED

=== StandardCNN, prune_ratio=40% ===
FP32 recovered: {'macro_f1': 0.8411750431929785, 'params': 22367, 'size_mb': 0.09412097930908203, 'latency_p50_ms': np.float64(0.12352499999224165), 'latency_p95_ms': np.float64(0.13241549999065683), 'latency_p99_ms': np.float64(0.14130119999549606)}


/tmp/ipykernel_1780/4132578582.py:52: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare(wrapped, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of 

PTQ: {'macro_f1': 0.7781568389754403, 'params': 22367, 'size_mb': 0.03797626495361328, 'latency_p50_ms': np.float64(0.18412500003250898), 'latency_p95_ms': np.float64(0.1961500000675187), 'latency_p99_ms': np.float64(0.19825799992645443)}
PTQ verdict (0.7782 vs bar 0.7819): BELOW RETENTION BAR
PTQ missed the bar -- building QAT fallback (same trigger rule as MA1DCNN's Step 5).


/tmp/ipykernel_1780/4132578582.py:91: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare_qat(wrapped, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release 

QAT: {'macro_f1': 0.8315259020278123, 'params': 22367, 'size_mb': 0.03797626495361328, 'latency_p50_ms': np.float64(0.18602000000100816), 'latency_p95_ms': np.float64(0.24028749998024054), 'latency_p99_ms': np.float64(0.31655809993253564)}
QAT verdict (0.8315 vs bar 0.7819): PASSED

=== StandardCNN, prune_ratio=60% ===
FP32 recovered: {'macro_f1': 0.8083053305992884, 'params': 11147, 'size_mb': 0.05127429962158203, 'latency_p50_ms': np.float64(0.10826000004726666), 'latency_p95_ms': np.float64(0.1152044999798818), 'latency_p99_ms': np.float64(0.12154239997812506)}


/tmp/ipykernel_1780/4132578582.py:52: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare(wrapped, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of 

PTQ: {'macro_f1': 0.7228520457679882, 'params': 11147, 'size_mb': 0.02534198760986328, 'latency_p50_ms': np.float64(0.19729500007770184), 'latency_p95_ms': np.float64(0.21087150003040733), 'latency_p99_ms': np.float64(0.2176275000579153)}
PTQ verdict (0.7229 vs bar 0.7819): BELOW RETENTION BAR
PTQ missed the bar -- building QAT fallback (same trigger rule as MA1DCNN's Step 5).


/tmp/ipykernel_1780/4132578582.py:91: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare_qat(wrapped, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release 

QAT: {'macro_f1': 0.8024010066525671, 'params': 11147, 'size_mb': 0.02534198760986328, 'latency_p50_ms': np.float64(0.201459999857434), 'latency_p95_ms': np.float64(0.2144614999338046), 'latency_p99_ms': np.float64(0.2177407000317544)}
QAT verdict (0.8024 vs bar 0.7819): PASSED

StandardCNN compression pipeline complete.
  20%: method=ptq  macro_f1=0.790884303847841
  40%: method=qat  macro_f1=0.8315259020278123
  60%: method=qat  macro_f1=0.8024010066525671


### 6.1 Controlled-comparison add-on -- force QAT on StandardCNN at 20% too

Section 6 let each architecture use the minimum compression method that cleared its own retention bar: StandardCNN passed on PTQ alone at 20%, but MA1DCNN's 20% checkpoint needed the QAT fallback. That means the Section 8 comparison has two things differing between checkpoints -- architecture AND how much fine-tuning each one got -- not architecture alone, which is what H1 is supposed to isolate.

This cell doesn't touch `standard_cnn_results` or anything Section 7.1/Section 8 already read from it -- those stay exactly as they were, as a real, valid (if caveated) result. It just builds one more, separately-named checkpoint: StandardCNN at 20%, forced through QAT regardless of whether PTQ already cleared the bar, reusing the already-recovered FP32 model Section 6 cached. Section 8.1 (after Section 8) then compares this against MA1DCNN's QAT checkpoint -- same compression method on both sides this time.

In [23]:
FORCED_QAT_RATIO = 0.20
forced_qat_path = f"{DATA_DIR}/standard_cnn_trial13_pruned{int(FORCED_QAT_RATIO*100)}_qat_forced.pt"

if os.path.exists(forced_qat_path):
    print("Found an existing forced-QAT StandardCNN checkpoint (20%) on Drive -- loading it.")
    standard_cnn_20_qat_forced = torch.load(forced_qat_path, weights_only=False, map_location="cpu")
else:
    # Reuses the recovered FP32 model Section 6 already pruned/fine-tuned
    # and cached -- no need to redo that expensive step, only the
    # quantization-side fine-tune changes here.
    recovered_20 = standard_cnn_results[FORCED_QAT_RATIO]["recovered_model"]
    standard_cnn_20_qat_forced, _ = quantize_standard_cnn_qat(
        recovered_20, TRIAL_13_TRAIN_CONFIG, train_loader, val_loader, device,
    )
    torch.save(standard_cnn_20_qat_forced, forced_qat_path)

forced_qat_bench = benchmark_variant(standard_cnn_20_qat_forced, val_loader, "cpu", TRIAL_13_TRAIN_CONFIG["threshold"])
print("StandardCNN 20%, forced QAT:", forced_qat_bench)

/tmp/ipykernel_1780/4132578582.py:91: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare_qat(wrapped, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release 

StandardCNN 20%, forced QAT: {'macro_f1': 0.834483207713564, 'params': 38192, 'size_mb': 0.05506610870361328, 'latency_p50_ms': np.float64(0.20444000006136775), 'latency_p95_ms': np.float64(0.22107000004325528), 'latency_p99_ms': np.float64(0.23165249990142908)}


## 7. `evaluation/noise_robustness.py` (h1_completion_plan.docx Section 6)

**[verbatim from h1_completion_plan.docx]**, cleaned of line-wrap
artifacts -- this is the one piece the plan gives complete, literal source
for, since nothing in this codebase previously added evaluation-time
noise. Once proven (Section 7.1 below), this belongs at
`src/edge_fault_dx/evaluation/noise_robustness.py`.


In [25]:
def add_snr_noise(X, snr_db, rng):
    """Adds zero-mean Gaussian noise to a window, scaled so the noisy
    signal sits at exactly snr_db decibels relative to X's own per-window
    power -- the standard SNR_dB = 10*log10(signal_power / noise_power)
    definition, inverted to solve for the noise variance that produces a
    given target SNR.

    X : np.ndarray, shape (28, 50) -- one already-normalized window
    """
    signal_power = np.mean(X ** 2)
    snr_linear = 10 ** (snr_db / 10)
    noise_power = signal_power / snr_linear
    noise = rng.normal(0.0, np.sqrt(noise_power), size=X.shape).astype(np.float32)
    return X + noise


class NoisyWindowedFaultDataset(torch.utils.data.Dataset):
    """Wraps an existing WindowedFaultDataset, adding independent
    add_snr_noise() noise to every window at a fixed SNR, with a given
    numpy Generator -- a new seed per trial is what makes repeated trials
    independent, not repeats of the identical noise draw.
    """
    def __init__(self, base_dataset, snr_db, seed):
        self.base = base_dataset
        self.snr_db = snr_db
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        X, y = self.base[idx]
        X_noisy = add_snr_noise(X.numpy(), self.snr_db, self.rng)
        return torch.from_numpy(X_noisy), y


SNR_LEVELS_DB = [0, 10, 20]      # H1's own testability spec
N_TRIALS = 5                      # H1's own minimum


def run_noise_robustness_trials(model, val_base_dataset, threshold, loss_fn, device):
    """Returns {snr_db: [trial_1_metrics, trial_2_metrics, ...]}. Clean
    metrics (no noise) are a separate baseline computed once via the
    existing evaluate() against val_base_dataset directly, kept outside
    this dict.

    NOTE: the doc's own literal signature includes loss_fn (matching
    evaluate()'s real required argument, confirmed against train.ipynb) --
    an earlier transcription of this cell dropped it while cleaning up
    the document's line-wrap artifacts; restored here.
    """
    results = {}
    for snr_db in SNR_LEVELS_DB:
        trial_metrics = []
        for trial in range(N_TRIALS):
            seed = hash((snr_db, trial)) % (2 ** 31)   # distinct, reproducible per (snr, trial)
            noisy_dataset = NoisyWindowedFaultDataset(val_base_dataset, snr_db, seed)
            noisy_loader = torch.utils.data.DataLoader(noisy_dataset, batch_size=256, shuffle=False)
            metrics = evaluate(model, noisy_loader, loss_fn, device, threshold)   # UNCHANGED function
            trial_metrics.append(metrics)
        results[snr_db] = trial_metrics
    return results


def summarize_trials(trial_metrics, metric_key="f1_macro"):
    """mean +/- 95% CI (t-distribution, small-N-appropriate for N_TRIALS=5)."""
    values = np.array([m[metric_key] for m in trial_metrics])
    mean = values.mean()
    sem = stats.sem(values)
    ci = sem * stats.t.ppf(0.975, df=len(values) - 1)
    return mean, ci


def paired_comparison(ma1dcnn_drops, standard_cnn_drops):
    """Paired t-test over matched (same snr_db, same trial index)
    accuracy-drop values -- the paired-test comparison H1's testability
    clause asks for. accuracy_drop[trial] = clean_f1_macro -
    noisy_f1_macro[trial], per model.
    """
    t_stat, p_value = stats.ttest_rel(ma1dcnn_drops, standard_cnn_drops)
    return t_stat, p_value


### 7.1 Self-check -- monotonic degradation on a synthetic/clean model

Per h1_completion_plan.docx Section 8, step 5: a model evaluated on its own
clean validation data should show monotonically falling macro-F1 as SNR
drops from 20 dB toward 0 dB, and near-chance performance at a
deliberately extreme, very low SNR -- the same "prove it standalone first"
discipline every other function in this pipeline was held to.


In [26]:
def _noise_robustness_self_check(model, val_base_dataset, threshold, device):
    extreme_snr_db = -20   # deliberately extreme, well outside SNR_LEVELS_DB
    check_levels = [extreme_snr_db, 0, 10, 20]
    scores = {}
    for snr_db in check_levels:
        seed = hash((snr_db, "selfcheck")) % (2 ** 31)
        noisy_dataset = NoisyWindowedFaultDataset(val_base_dataset, snr_db, seed)
        noisy_loader = torch.utils.data.DataLoader(noisy_dataset, batch_size=256, shuffle=False)
        scores[snr_db] = evaluate(model, noisy_loader, loss_fn, device, threshold)["f1_macro"]
        print(f"  snr={snr_db:>4} dB  macro_f1={scores[snr_db]:.4f}")

    assert scores[0] <= scores[10] <= scores[20], (
        f"expected monotonically falling macro_f1 as SNR drops (20>=10>=0), got {scores}"
    )
    assert scores[extreme_snr_db] < scores[0], (
        f"expected near-chance degradation at extreme low SNR ({extreme_snr_db} dB) to score "
        f"below the 0 dB level, got {scores}"
    )
    print("Self-check PASSED: macro_f1 degrades monotonically as SNR drops.")

print("Running noise-robustness self-check against the recovered FP32 StandardCNN (20% ratio):")
_noise_robustness_self_check(
    standard_cnn_results[0.20]["recovered_model"],
    val_loader.dataset,
    TRIAL_13_TRAIN_CONFIG["threshold"],
    device,
)


Running noise-robustness self-check against the recovered FP32 StandardCNN (20% ratio):
  snr= -20 dB  macro_f1=0.2364
  snr=   0 dB  macro_f1=0.4848
  snr=  10 dB  macro_f1=0.6942
  snr=  20 dB  macro_f1=0.8142
Self-check PASSED: macro_f1 degrades monotonically as SNR drops.


## 8. The real paired comparison (h1_completion_plan.docx Section 8, step 6)

Pick each architecture's best-performing compressed checkpoint from
Section 6 above, run both through `run_noise_robustness_trials` at
0/10/20 dB x 5 trials, then `summarize_trials` + `paired_comparison` to
produce H1's actual reported numbers. **Set `CHOSEN_RATIO` to whichever
pruning ratio you're comparing** -- the plan doesn't name one in advance;
it's a choice you make from Section 6's real results once you have them.


In [27]:
from edge_fault_dx.compression.quantization import QuantizedConvTrunk
from edge_fault_dx.models.ma1dcnn import ChannelAttentionModule, ElementwiseAttentionModule

CHOSEN_RATIO = 0.20   # TODO: set this from your real Section 6 results

# REAL BUG, found by actually running this: AttributeError: Can't get
# attribute 'QuantizedConvTrunk' on <module '__main__'>. torch.save(model, ...)
# pickles a class reference alongside the weights -- the module the class
# was DEFINED in at save time. This checkpoint was produced by an earlier
# notebook where QuantizedConvTrunk was defined in a cell (so pickle
# recorded its module as "__main__"), and this notebook's __main__ has no
# class of that name. Fixed by importing the real QuantizedConvTrunk from
# quantization.py under that exact name right here -- top-level notebook
# code runs as __main__, so this binds the name where the unpickler looks.
from edge_fault_dx.compression.quantization import QuantizedConvTrunk

# weights_only=False: PyTorch 2.6+ defaults torch.load to a safer mode that
# refuses to unpickle custom classes unless told explicitly they're
# trusted. This checkpoint is one this project generated itself, so
# that's fine.
#
# map_location="cpu": this is a QUANTIZED checkpoint -- eager-mode
# quantization (fbgemm) only has CPU kernels, so it has to be loaded and
# evaluated on CPU regardless of what device the rest of the notebook
# trained FP32 models on.
ma1dcnn_compressed = torch.load(
    f"{DATA_DIR}/ma1dcnn_trial13_pruned{int(CHOSEN_RATIO*100)}_quantized_qat.pt",
    weights_only=False,
    map_location="cpu",
)
ma1dcnn_compressed = repair_missing_module_dicts(ma1dcnn_compressed)   # cross-process reload fix, Section 12.2

standard_cnn_compressed = standard_cnn_results[CHOSEN_RATIO]["best_model"]

clean_threshold = TRIAL_13_TRAIN_CONFIG["threshold"]

ma1dcnn_clean = evaluate(ma1dcnn_compressed, val_loader, loss_fn, "cpu", threshold=clean_threshold)
standard_cnn_clean = evaluate(standard_cnn_compressed, val_loader, loss_fn, "cpu", threshold=clean_threshold)
print("clean (no noise) macro_f1 -- MA1DCNN:", ma1dcnn_clean["f1_macro"], " StandardCNN:", standard_cnn_clean["f1_macro"])

ma1dcnn_noise_results = run_noise_robustness_trials(ma1dcnn_compressed, val_loader.dataset, clean_threshold, loss_fn, "cpu")
standard_cnn_noise_results = run_noise_robustness_trials(standard_cnn_compressed, val_loader.dataset, clean_threshold, loss_fn, "cpu")

print(f"\n{'SNR (dB)':>10} | {'MA1DCNN mean':>14} ± {'95% CI':>8} | {'StandardCNN mean':>18} ± {'95% CI':>8}")
for snr_db in SNR_LEVELS_DB:
    ma1dcnn_mean, ma1dcnn_ci = summarize_trials(ma1dcnn_noise_results[snr_db])
    standard_cnn_mean, standard_cnn_ci = summarize_trials(standard_cnn_noise_results[snr_db])
    print(f"{snr_db:>10} | {ma1dcnn_mean:>14.4f} ± {ma1dcnn_ci:>8.4f} | {standard_cnn_mean:>18.4f} ± {standard_cnn_ci:>8.4f}")

# paired accuracy-drop comparison, matched (snr_db, trial) pairs
ma1dcnn_drops = [
    ma1dcnn_clean["f1_macro"] - m["f1_macro"]
    for snr_db in SNR_LEVELS_DB for m in ma1dcnn_noise_results[snr_db]
]
standard_cnn_drops = [
    standard_cnn_clean["f1_macro"] - m["f1_macro"]
    for snr_db in SNR_LEVELS_DB for m in standard_cnn_noise_results[snr_db]
]
t_stat, p_value = paired_comparison(ma1dcnn_drops, standard_cnn_drops)
print(f"\npaired t-test on accuracy-drop (MA1DCNN vs StandardCNN): t={t_stat:.4f}, p={p_value:.4f}")

clean (no noise) macro_f1 -- MA1DCNN: 0.8382103657364457  StandardCNN: 0.790884303847841

  SNR (dB) |   MA1DCNN mean ±   95% CI |   StandardCNN mean ±   95% CI
         0 |         0.4747 ±   0.0013 |             0.4851 ±   0.0010
        10 |         0.7159 ±   0.0004 |             0.6925 ±   0.0010
        20 |         0.8262 ±   0.0005 |             0.8111 ±   0.0006

paired t-test on accuracy-drop (MA1DCNN vs StandardCNN): t=9.8485, p=0.0000


In [28]:
 # ### 8.1 Controlled comparison -- same compression method on both sides
#
# Section 8 above compared MA1DCNN (QAT) against StandardCNN (PTQ-only --
# it cleared its own bar without needing QAT), so that result has
# compression depth confounded with architecture. This cell re-runs the
# same comparison using standard_cnn_20_qat_forced from Section 6.1
# instead -- QAT on both sides -- so architecture is the only thing left
# differing between the two checkpoints.
#
# MA1DCNN's clean/noise numbers are NOT recomputed here -- ma1dcnn_clean,
# ma1dcnn_noise_results, and ma1dcnn_drops from Section 8 are reused
# as-is, since that checkpoint was always QAT and nothing about it
# changes in this cell. Only StandardCNN's side is computed fresh,
# against the forced-QAT checkpoint.

standard_cnn_clean_controlled = evaluate(
    standard_cnn_20_qat_forced, val_loader, loss_fn, "cpu", threshold=clean_threshold,
)
print("[controlled, both QAT] clean (no noise) macro_f1 -- MA1DCNN:", ma1dcnn_clean["f1_macro"],
      " StandardCNN:", standard_cnn_clean_controlled["f1_macro"])

standard_cnn_noise_results_controlled = run_noise_robustness_trials(
    standard_cnn_20_qat_forced, val_loader.dataset, clean_threshold, loss_fn, "cpu",
)

print(f"\n{'SNR (dB)':>10} | {'MA1DCNN mean':>14} ± {'95% CI':>8} | {'StandardCNN mean':>18} ± {'95% CI':>8}")
for snr_db in SNR_LEVELS_DB:
    ma1dcnn_mean, ma1dcnn_ci = summarize_trials(ma1dcnn_noise_results[snr_db])
    standard_cnn_mean, standard_cnn_ci = summarize_trials(standard_cnn_noise_results_controlled[snr_db])
    print(f"{snr_db:>10} | {ma1dcnn_mean:>14.4f} ± {ma1dcnn_ci:>8.4f} | {standard_cnn_mean:>18.4f} ± {standard_cnn_ci:>8.4f}")

standard_cnn_drops_controlled = [
    standard_cnn_clean_controlled["f1_macro"] - m["f1_macro"]
    for snr_db in SNR_LEVELS_DB for m in standard_cnn_noise_results_controlled[snr_db]
]
t_stat_controlled, p_value_controlled = paired_comparison(ma1dcnn_drops, standard_cnn_drops_controlled)
print(f"\n[controlled -- both QAT] paired t-test on accuracy-drop "
      f"(MA1DCNN vs StandardCNN): t={t_stat_controlled:.4f}, p={p_value_controlled:.4f}")

[controlled, both QAT] clean (no noise) macro_f1 -- MA1DCNN: 0.8382103657364457  StandardCNN: 0.834483207713564

  SNR (dB) |   MA1DCNN mean ±   95% CI |   StandardCNN mean ±   95% CI
         0 |         0.4747 ±   0.0013 |             0.4858 ±   0.0014
        10 |         0.7159 ±   0.0004 |             0.7015 ±   0.0006
        20 |         0.8262 ±   0.0005 |             0.8207 ±   0.0007

[controlled -- both QAT] paired t-test on accuracy-drop (MA1DCNN vs StandardCNN): t=0.2834, p=0.7810


## 9. What this notebook does and does not settle (h1_completion_plan.docx Section 9)

Running Sections 1-8 for real gives a genuine, architecture-controlled
answer to whether MA1DCNN's attention modules earn their added parameters
and compute, under compression and under noise.

**What it does not settle on its own:** the uncompressed MA1DCNN baseline
(0.8401 macro-F1) is already below H1's 93.25% floor relative to
Akindoju's (2025) 98.16% -- an upstream gap this notebook does not touch
and the StandardCNN comparison cannot close, since it's a property of this
project's own confirmed baseline, not of compression or of which
architecture is compressed. And everything here runs on Colab CPU/GPU,
which is sufficient for the accuracy/precision/robustness comparison
itself, but not a substitute for the on-device confirmation H1's literal
"deployed on Jetson" wording calls for once Phase 6 exists.
